# Temporal Analysis
`capabilities/temporal/growth_tracker.py`  
`capabilities/temporal/change_detector.py`

Turn individual snapshots into a continuous growth story.

| Class | Purpose | Speed |
|-------|---------|-------|
| `GrowthTracker` | Accumulate metrics over time, detect trends and alerts | No computation |
| `ChangeDetector` | Detect and quantify visual change between two frames | < 20ms / pair |

`GrowthTracker` consumes results from other capabilities — it does no computer vision itself.

## Setup

In [ ]:
import sys, os, tempfile, json
sys.path.insert(0, os.path.abspath('../..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from capabilities.temporal.growth_tracker import GrowthTracker
from capabilities.temporal.change_detector import ChangeDetector
from capabilities.analysis.coverage_estimator import CoverageEstimator
from sprout_detection.utils.image_gen import make_sprout_image, make_bare_soil_image

tmp = tempfile.mkdtemp()
print('Ready')

---
# Part 1 — GrowthTracker

## Create a growth log

In [ ]:
tracker = GrowthTracker(plant_id='pot_03')

# Simulate 10 daily observations by building mock results
# In production you'd use real CoverageEstimator / DepthEstimator results

# Growth curve: germination → rapid growth → stress at day 7 → recovery
mock_data = [
    # (day, green%, health_ratio, anomaly_score)
    (0,  2.1,  0.90, 0.10),
    (1,  5.4,  0.92, 0.09),
    (2, 12.3,  0.94, 0.08),
    (3, 22.8,  0.91, 0.11),
    (4, 35.2,  0.89, 0.12),
    (5, 44.1,  0.88, 0.14),
    (6, 48.7,  0.82, 0.20),
    (7, 42.3,  0.55, 0.65),  # ← stress event: yellow_ratio increased
    (8, 38.1,  0.51, 0.70),  # ← still stressed
    (9, 40.2,  0.58, 0.60),  # ← beginning recovery
]

for day, green_pct, health, anomaly in mock_data:
    # log_manual lets you record any subset of fields directly
    tracker.log_manual(
        timestamp=f'2025-06-{day+1:02d}',
        green_coverage_pct=green_pct,
        total_coverage_pct=green_pct * 1.12,
        health_ratio=health,
        anomaly_score=anomaly,
    )

print(f'Tracker has {tracker.record_count} records')

## Analytics — growth rate and health trend

In [ ]:
growth_rate  = tracker.growth_rate()
health_trend = tracker.health_trend()

print(f'Growth rate (avg Δ green/day): {growth_rate:+.2f}%')
print(f'Health trend (slope):          {health_trend:+.6f}')
print()
if growth_rate and growth_rate > 0:
    print(f'  → Plant is growing at {growth_rate:.1f}% green coverage per day')
if health_trend and health_trend < -0.01:
    print(f'  → Health is declining — investigate stress source')

## Automatic alert detection

In [ ]:
alerts = tracker.get_alerts()

if alerts:
    print(f'{len(alerts)} alert(s) detected:\n')
    for a in alerts:
        icon = '🔴' if a['severity'] == 'HIGH' else '🟡'
        print(f'  {icon} [{a["severity"]}] {a["type"]}')
        print(f'     {a["message"]}')
        print()
else:
    print('No alerts triggered')

## plot() — full multi-panel growth chart

In [ ]:
tracker.plot(title='Pot 03 — 10-Day Growth Log')

## Logging real capability results

In [ ]:
# In production, log results from actual capability classes:

estimator = CoverageEstimator()

# Generate a fresh image
img_path = os.path.join(tmp, 'today.png')
cv2.imwrite(img_path, make_sprout_image(seed=99))

tracker2 = GrowthTracker(plant_id='pot_07')

# Log a real coverage result
coverage_result = estimator.estimate(img_path)
record = tracker2.log_coverage(coverage_result, timestamp='2025-06-10')

print('Logged coverage result:')
print(f'  green_coverage_pct: {record.green_coverage_pct:.2f}%')
print(f'  health_ratio:       {record.health_ratio:.3f}')
print(f'  timestamp:          {record.timestamp}')

## Save and load a growth log

In [ ]:
log_path = os.path.join(tmp, 'pot_03_log.json')

# Save
tracker.save(log_path)

# Load into a new tracker
tracker_loaded = GrowthTracker()
tracker_loaded.load(log_path)

print(f'Loaded {tracker_loaded.record_count} records')
print(f'Plant ID: {tracker_loaded.plant_id}')

# Preview the JSON
with open(log_path) as f:
    data = json.load(f)
print(f'\nFirst record:')
print(json.dumps(data['records'][0], indent=2))

---
# Part 2 — ChangeDetector

## Generate image pairs for comparison

In [ ]:
detector = ChangeDetector()

# Create pairs: same plant, then stressed, then very different
path_a   = os.path.join(tmp, 'day_a.png')
path_b_small   = os.path.join(tmp, 'day_b_growth.png')   # slight growth
path_b_wilting = os.path.join(tmp, 'day_b_wilt.png')     # wilting
path_b_bare    = os.path.join(tmp, 'day_b_bare.png')     # bare soil

cv2.imwrite(path_a,         make_sprout_image(seed=10))

# Slight growth: same image with a bit more green
img_growth = make_sprout_image(seed=10)
hsv = cv2.cvtColor(img_growth, cv2.COLOR_BGR2HSV).astype(np.float32)
hsv[:,:,1] = np.clip(hsv[:,:,1] * 1.1, 0, 255)
cv2.imwrite(path_b_small, cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR))

# Wilting: shift to yellow
img_wilt = make_sprout_image(seed=10)
hsv = cv2.cvtColor(img_wilt, cv2.COLOR_BGR2HSV).astype(np.float32)
hsv[:,:,0] = np.clip(hsv[:,:,0] - 20, 0, 179)
hsv[:,:,1] = np.clip(hsv[:,:,1] * 0.6, 0, 255)
cv2.imwrite(path_b_wilting, cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR))

cv2.imwrite(path_b_bare, make_bare_soil_image(seed=10))

print('Image pairs created')

## compare() — detect change between two images

In [ ]:
cases = [
    ('Growth',        path_b_small),
    ('Wilting',       path_b_wilting),
    ('Plant removed', path_b_bare),
]

results = []
for label, path_b in cases:
    r = detector.compare(path_a, path_b)
    results.append((label, r))
    print(f'{label:<16}: type={r.change_type:<10}  magnitude={r.change_magnitude:.3f}  '
          f'pixel_diff={r.pixel_diff_ratio:.3f}  green_delta={r.green_delta:+.3f}')

## Visualise diff maps

In [ ]:
n = len(cases)
fig, axes = plt.subplots(3, n, figsize=(5*n, 11))

img_a_rgb = cv2.cvtColor(cv2.imread(path_a), cv2.COLOR_BGR2RGB)

for col, (label, (_, path_b)) in enumerate(zip(results, [(l, p) for l, p in cases])):
    # Row 0: image A
    axes[0, col].imshow(img_a_rgb)
    axes[0, col].set_title(f'Before\n(Day A)', fontsize=9)
    axes[0, col].axis('off')
    
    # Row 1: image B
    img_b_rgb = cv2.cvtColor(cv2.imread(path_b), cv2.COLOR_BGR2RGB)
    axes[1, col].imshow(img_b_rgb)
    axes[1, col].set_title(f'After\n({label})', fontsize=9)
    axes[1, col].axis('off')
    
    # Row 2: diff map
    _, r = results[col]
    if r.diff_map is not None:
        axes[2, col].imshow(r.diff_map, cmap='hot')
        axes[2, col].set_title(
            f'Diff map\ntype={r.change_type}  Δ={r.change_magnitude:.3f}', fontsize=8
        )
    axes[2, col].axis('off')

plt.suptitle('ChangeDetector — Diff Maps', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## compare_series() — track changes across a full time series

In [ ]:
# Create a 6-image time series
series_imgs = []
for i in range(6):
    p = os.path.join(tmp, f'series_{i:02d}.png')
    img = make_sprout_image(seed=i)
    if i >= 4:
        # Simulate stress after day 4
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:,:,0] = np.clip(hsv[:,:,0] - (i-3)*10, 0, 179)
        img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    cv2.imwrite(p, img)
    series_imgs.append(p)

change_series = detector.compare_series(series_imgs)

print(f'Processed {len(change_series)} consecutive pairs:\n')
print(f'{"Pair":<8} {"Type":<12} {"Magnitude":<12} {"Green Δ":<10} {"Time"}')
print('-' * 55)
for i, r in enumerate(change_series):
    print(f'{i}→{i+1:<5} {r.change_type:<12} {r.change_magnitude:<12.3f} '
          f'{r.green_delta:>+8.3f}   {r.duration_ms:.0f}ms')

## Feed ChangeDetector results into GrowthTracker

In [ ]:
# Full pipeline: coverage + change → unified tracker
coverage_est = CoverageEstimator()
tracker3 = GrowthTracker(plant_id='series_demo')

for i, path in enumerate(series_imgs):
    cov = coverage_est.estimate(path)
    tracker3.log_coverage(cov, timestamp=f'2025-06-{i+1:02d}')

print(f'Tracker records: {tracker3.record_count}')
print(f'Growth rate    : {tracker3.growth_rate():+.2f}% per day')
print(f'Health trend   : {tracker3.health_trend():+.6f}')

tracker3.plot(title='Series Demo — Coverage + Health Trend')